# 06 — Async Client Smoke + No-Leakage Audit

`AsyncNostradamusClient` parity + assert no notebook projects are left behind.

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" python-dotenv

In [ ]:
import asyncio, os, random, time
from dotenv import load_dotenv
load_dotenv()
BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
assert BASE_URL and USERNAME and PASSWORD, "SKIP: set CENOTOO_BASE_URL/USERNAME/PASSWORD"
def unique_name(p="nb"):
    return f"{p}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}"

In [ ]:
from nostradamus_ioto_sdk import NostradamusClient, AsyncNostradamusClient
sync = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
probe = sync.projects.create(name=unique_name("nb6probe"), description="nb06", tags=["notebook", "06"])
print("probe project:", probe.project_id)
async def _main():
    async with AsyncNostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD) as c:
        got = await c.projects.aget(probe.project_id)
        assert str(got.project_id) == str(probe.project_id)
        print("async aget OK:", got.name)
asyncio.run(_main())

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    sync.projects.delete(probe.project_id)
    print("probe deleted")
# Audit: no notebook leftovers (prefix nb / nb2 / nbmqtt / nbcoap / pipe)
left = [p for p in sync.projects.list() if str(getattr(p, 'name', '')).startswith(("nb", "pipe"))]
print("leftover notebook projects:", len(left))
assert len(left) == 0, [p.name for p in left]
sync.close()
print("audit OK: no leakage")